# 🔑 DINO SDK Token Extractor - Databricks Notebook

Este notebook extrai tokens de administração e configurações do workspace Databricks para uso no DINO SDK.
Ele é executado automaticamente pelo WHL quando tokens são necessários.

**Funcionalidades:**
- Extrai admin token do contexto Databricks
- Obtém URL do workspace
- Cria headers de autorização
- Retorna dados em formato JSON para uso externo

In [ ]:
# Section: Import Required Libraries
# Import necessary libraries including Databricks utilities and error handling modules

import json
import logging
from datetime import datetime
from typing import Dict, Any, Optional

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("🔧 DINO SDK Token Extractor v1.1.4")
print("=" * 50)
print("📅 Executado em:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("🎯 Objetivo: Extrair tokens e configurações Databricks")

In [ ]:
# Section: Define Global Variables
# Initialize global variables that will store authentication tokens and instance URLs

# Variáveis globais para armazenar tokens e configurações
admin_token = None
databricks_instance = None
workspace_url = None
headers = None
extraction_result = {}

print("\n🔧 Inicializando variáveis globais...")
print("✅ Variáveis globais criadas:")
print("   - admin_token: None")
print("   - databricks_instance: None")
print("   - workspace_url: None")
print("   - headers: None")
print("   - extraction_result: {}")

In [ ]:
# Section: Retrieve Admin Token from Databricks Context
# Use dbutils to extract the admin token from the Databricks notebook context

print("\n🔑 FASE 1: Extraindo Admin Token...")

try:
    # Extrair token do contexto Databricks usando dbutils
    admin_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    
    if admin_token:
        # Mascarar token para logs de segurança
        masked_token = f"{admin_token[:15]}...{admin_token[-8:]}" if len(admin_token) > 23 else "***"
        print(f"✅ Admin token extraído com sucesso: {masked_token}")
        print(f"📏 Tamanho do token: {len(admin_token)} caracteres")
        
        # Validar formato básico do token
        if admin_token.startswith('dapi') and len(admin_token) > 30:
            print("✅ Token possui formato válido (dapi...)")
        else:
            print("⚠️ Token pode não ter formato esperado")
            
    else:
        print("❌ Token extraído está vazio")
        raise ValueError("Token de administração não encontrado")
        
except Exception as e:
    print(f"❌ Erro ao extrair admin token: {e}")
    print("💡 Verifique se o notebook está executando em ambiente Databricks válido")
    raise

In [ ]:
# Section: Extract Databricks Instance URL
# Get the Databricks workspace URL from the notebook context using dbutils methods

print("\n🌐 FASE 2: Extraindo URL do Workspace...")

try:
    # Extrair instância/URL do workspace
    databricks_instance = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    
    if databricks_instance:
        print(f"✅ Instância Databricks extraída: {databricks_instance}")
        
        # Garantir formato HTTPS correto
        if not databricks_instance.startswith('https://'):
            workspace_url = f"https://{databricks_instance}"
            print(f"🔧 Adicionado https:// -> {workspace_url}")
        else:
            workspace_url = databricks_instance
            print(f"✅ URL já no formato correto: {workspace_url}")
        
        # Validar formato da URL
        if '.azuredatabricks.net' in workspace_url or '.databricks.com' in workspace_url:
            print("✅ URL possui formato Databricks válido")
        else:
            print("⚠️ URL pode não ser um workspace Databricks padrão")
            
    else:
        print("❌ URL do workspace extraída está vazia")
        raise ValueError("URL do workspace não encontrada")
        
except Exception as e:
    print(f"❌ Erro ao extrair URL do workspace: {e}")
    print("💡 Verifique se o contexto Databricks está disponível")
    raise

In [ ]:
# Section: Create Authorization Headers
# Format the retrieved token into proper authorization headers for API requests

print("\n🔐 FASE 3: Criando Headers de Autorização...")

try:
    # Criar headers de autorização
    headers = {
        "Authorization": f"Bearer {admin_token}",
        "Content-Type": "application/json",
        "User-Agent": "DINO-SDK/1.1.4"
    }
    
    print("✅ Headers de autorização criados:")
    print(f"   - Authorization: Bearer {admin_token[:15]}...")
    print(f"   - Content-Type: application/json")
    print(f"   - User-Agent: DINO-SDK/1.1.4")
    
    # Validar estrutura dos headers
    if "Bearer" in headers["Authorization"] and len(headers["Authorization"]) > 50:
        print("✅ Headers possuem formato válido")
    else:
        print("⚠️ Headers podem estar mal formatados")
        
except Exception as e:
    print(f"❌ Erro ao criar headers: {e}")
    raise

In [ ]:
# Section: Implement Notebook Runner Class
# Create a NotebookRunner class that can execute other notebooks using dbutils.notebook.run

print("\n🔧 FASE 4: Preparando Funcionalidades do Notebook Runner...")

class DatabricksNotebookRunner:
    """
    Classe para executar notebooks e gerenciar contexto Databricks
    """
    
    def __init__(self, timeout: int = 300):
        self.timeout = timeout
        self.logger = logging.getLogger(__name__)
    
    def run_notebook(self, path: str, params: Dict[str, str] = None) -> str:
        """Executa um notebook Databricks"""
        if params is None:
            params = {}
        
        try:
            self.logger.info(f"🚀 Executando notebook: {path}")
            result = dbutils.notebook.run(path, self.timeout, params)
            self.logger.info(f"✅ Notebook executado com sucesso")
            return result
        except Exception as e:
            self.logger.error(f"❌ Erro ao executar notebook {path}: {e}")
            raise RuntimeError(f"Erro ao executar notebook {path}: {e}")
    
    @staticmethod
    def get_current_context():
        """Retorna contexto atual do notebook"""
        return {
            'admin_token': admin_token,
            'workspace_url': workspace_url,
            'databricks_instance': databricks_instance,
            'headers': headers
        }

# Instanciar runner para uso
notebook_runner = DatabricksNotebookRunner()

print("✅ Classe NotebookRunner criada e instanciada")
print("📋 Funcionalidades disponíveis:")
print("   - run_notebook(path, params)")
print("   - get_current_context()")

In [ ]:
# Section: Test Token Retrieval Function
# Implement validation functions to test the token retrieval process

print("\n🧪 FASE 5: Validando Extração de Tokens...")

def validate_extraction() -> Dict[str, Any]:
    """
    Valida se a extração foi bem-sucedida e retorna dados estruturados
    """
    validation_results = {
        'success': False,
        'admin_token_valid': False,
        'workspace_url_valid': False,
        'headers_valid': False,
        'errors': []
    }
    
    # Validar admin token
    if admin_token and isinstance(admin_token, str) and len(admin_token) > 30:
        validation_results['admin_token_valid'] = True
        print("✅ Admin token válido")
    else:
        validation_results['errors'].append("Admin token inválido ou ausente")
        print("❌ Admin token inválido")
    
    # Validar workspace URL
    if workspace_url and isinstance(workspace_url, str) and workspace_url.startswith('https://'):
        validation_results['workspace_url_valid'] = True
        print("✅ Workspace URL válida")
    else:
        validation_results['errors'].append("Workspace URL inválida ou ausente")
        print("❌ Workspace URL inválida")
    
    # Validar headers
    if headers and isinstance(headers, dict) and 'Authorization' in headers:
        validation_results['headers_valid'] = True
        print("✅ Headers válidos")
    else:
        validation_results['errors'].append("Headers inválidos ou ausentes")
        print("❌ Headers inválidos")
    
    # Determinar sucesso geral
    validation_results['success'] = (
        validation_results['admin_token_valid'] and 
        validation_results['workspace_url_valid'] and 
        validation_results['headers_valid']
    )
    
    return validation_results

# Executar validação
validation = validate_extraction()

if validation['success']:
    print("\n🎉 VALIDAÇÃO COMPLETA COM SUCESSO!")
else:
    print(f"\n❌ VALIDAÇÃO FALHOU: {', '.join(validation['errors'])}")

In [ ]:
# Criar resultado final estruturado para retorno
extraction_result = {
    'admin_token': admin_token,
    'workspace_url': workspace_url,
    'databricks_instance': databricks_instance,
    'headers': headers,
    'extraction_method': 'databricks_notebook',
    'timestamp': datetime.now().isoformat(),
    'validation': validation,
    'notebook_runner_available': True
}

print("\n📦 RESULTADO FINAL:")
print("=" * 50)

# Mostrar resumo sem expor token completo
summary = {
    'admin_token_length': len(admin_token) if admin_token else 0,
    'workspace_url': workspace_url,
    'headers_count': len(headers) if headers else 0,
    'extraction_success': validation['success'],
    'timestamp': extraction_result['timestamp']
}

for key, value in summary.items():
    print(f"   {key}: {value}")

# Retornar JSON para uso externo (o notebook runner vai capturar isto)
if validation['success']:
    result_json = json.dumps(extraction_result, default=str)
    print(f"\n✅ JSON gerado com {len(result_json)} caracteres")
    print("🎯 Dados prontos para uso no DINO SDK")
    
    # dbutils.notebook.exit() retornará este JSON
    dbutils.notebook.exit(result_json)
else:
    error_result = {
        'success': False,
        'errors': validation['errors'],
        'timestamp': datetime.now().isoformat()
    }
    error_json = json.dumps(error_result)
    print(f"\n❌ Retornando erro: {error_json}")
    dbutils.notebook.exit(error_json)